# TorchGeo FTW segmentation -> contourrs polygons

This notebook runs `Unet_Weights.SENTINEL2_FTW_PRUE_CCBY_EFNETB3` on a Fields of the World sample and polygonizes only class index `1` (field) with `contourrs.shapes`.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import torch
from contourrs import shapes
from torchgeo.datasets import FieldsOfTheWorld
from torchgeo.models import Unet_Weights, unet

In [ ]:
def preprocess(sample):
    sample["image"] = sample["image"] / 3000.0
    return sample

ds = FieldsOfTheWorld(
    root="data",
    split="test",
    target="3-class",
    countries="spain",
    transforms=preprocess,
    download=False,
)

idx = 50
sample = ds[idx]

In [ ]:
model = unet(weights=Unet_Weights.SENTINEL2_FTW_PRUE_CCBY_EFNETB3).eval()

with torch.inference_mode():
    logits = model(sample["image"].unsqueeze(0))

prediction = logits.squeeze(0).argmax(dim=0).cpu().numpy().astype(np.uint8)
prediction.shape

In [ ]:
class_index = 1
class_mask = prediction == class_index

features = [
    {
        "type": "Feature",
        "properties": {"class_idx": int(value)},
        "geometry": geometry,
    }
    for geometry, value in shapes(
        class_mask.astype(np.uint8),
        mask=class_mask,
        connectivity=4,
    )
    if int(value) == 1
]

field_polygons = gpd.GeoDataFrame.from_features(features)
field_polygons.head()

In [ ]:
image = sample["image"].cpu().numpy()
rgb = np.clip(np.moveaxis(image[:3], 0, -1), 0.0, 1.0)
h, w = prediction.shape

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(rgb)
axes[0].set_title("Sentinel-2 RGB")

axes[1].imshow(class_mask, cmap="gray")
axes[1].set_title("Class 1 mask")

axes[2].imshow(rgb)
field_polygons.boundary.plot(ax=axes[2], color="#ffcc00", linewidth=0.6)
axes[2].set_title(f"Class 1 polygons ({len(field_polygons):,})")

for ax in axes:
    ax.set_axis_off()
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

output_path = Path("examples/output/ftw_fields_idx50.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)
field_polygons.to_parquet(output_path, index=False)
output_path